## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Customer churn in matched signup-week sets

**Conditional Logistic Regression** · a worked example from the StatNex teaching guide

**The story.** A phone company formed 29 matched sets of six customers who signed up in the same week: in each set one customer later cancelled (churned) and five stayed. Matching on the signup week holds that week's promotions, prices and season fixed within each set.

**The question.** Among customers who joined at the same time, what separates the one who left from the five who stayed?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `09_multivariate/conditional_logistic_regression.csv` (174 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
No.,Matched set,Churned,Monthly bill ($),Filed complaint,Contacted support,Annual contract
1.0,1.0,0.0,55.0,0.0,1.0,0.0
2.0,1.0,0.0,78.0,0.0,0.0,0.0
3.0,1.0,0.0,30.0,0.0,1.0,0.0
4.0,1.0,0.0,38.0,0.0,0.0,1.0
5.0,1.0,1.0,70.0,0.0,1.0,0.0
6.0,1.0,0.0,45.0,0.0,0.0,1.0
7.0,2.0,0.0,49.0,0.0,1.0,1.0
8.0,2.0,0.0,75.0,0.0,1.0,1.0
9.0,2.0,0.0,49.0,0.0,0.0,1.0
10.0,2.0,0.0,49.0,0.0,1.0,0.0
11.0,2.0,0.0,51.0,0.0,0.0,1.0
12.0,2.0,1.0,61.0,1.0,0.0,1.0
13.0,3.0,0.0,59.0,0.0,0.0,0.0
14.0,3.0,0.0,32.0,0.0,0.0,0.0
15.0,3.0,0.0,54.0,0.0,0.0,0.0
16.0,3.0,0.0,48.0,0.0,0.0,1.0
17.0,3.0,0.0,78.0,0.0,0.0,1.0
18.0,3.0,1.0,50.0,0.0,0.0,0.0
19.0,4.0,0.0,45.0,0.0,0.0,0.0
20.0,4.0,0.0,60.0,1.0,0.0,1.0
21.0,4.0,0.0,49.0,0.0,0.0,1.0
22.0,4.0,1.0,61.0,1.0,1.0,0.0
23.0,4.0,0.0,47.0,0.0,0.0,1.0
24.0,4.0,0.0,35.0,0.0,0.0,1.0
25.0,5.0,0.0,52.0,0.0,1.0,0.0
26.0,5.0,0.0,37.0,0.0,0.0,1.0
27.0,5.0,0.0,48.0,1.0,0.0,0.0
28.0,5.0,0.0,39.0,0.0,0.0,1.0
29.0,5.0,0.0,59.0,0.0,0.0,0.0
30.0,5.0,1.0,80.0,1.0,0.0,0.0
31.0,6.0,0.0,53.0,0.0,0.0,1.0
32.0,6.0,0.0,64.0,0.0,0.0,1.0
33.0,6.0,0.0,58.0,0.0,0.0,0.0
34.0,6.0,0.0,51.0,0.0,0.0,0.0
35.0,6.0,1.0,61.0,1.0,1.0,0.0
36.0,6.0,0.0,58.0,0.0,0.0,1.0
37.0,7.0,0.0,50.0,0.0,0.0,1.0
38.0,7.0,0.0,49.0,0.0,0.0,0.0
39.0,7.0,0.0,35.0,0.0,0.0,0.0
40.0,7.0,1.0,42.0,0.0,1.0,0.0
41.0,7.0,0.0,69.0,0.0,1.0,0.0
42.0,7.0,0.0,26.0,0.0,0.0,1.0
43.0,8.0,0.0,43.0,0.0,0.0,0.0
44.0,8.0,0.0,51.0,0.0,0.0,0.0
45.0,8.0,1.0,45.0,0.0,1.0,0.0
46.0,8.0,0.0,49.0,0.0,1.0,0.0
47.0,8.0,0.0,46.0,0.0,0.0,1.0
48.0,8.0,0.0,63.0,0.0,0.0,0.0
49.0,9.0,0.0,28.0,1.0,0.0,0.0
50.0,9.0,0.0,32.0,0.0,0.0,0.0
51.0,9.0,0.0,15.0,0.0,1.0,0.0
52.0,9.0,0.0,54.0,0.0,0.0,0.0
53.0,9.0,0.0,25.0,0.0,0.0,0.0
54.0,9.0,1.0,47.0,1.0,0.0,1.0
55.0,10.0,0.0,58.0,0.0,0.0,0.0
56.0,10.0,1.0,50.0,1.0,0.0,0.0
57.0,10.0,0.0,36.0,0.0,0.0,0.0
58.0,10.0,0.0,70.0,0.0,1.0,0.0
59.0,10.0,0.0,43.0,0.0,0.0,0.0
60.0,10.0,0.0,80.0,0.0,1.0,0.0
61.0,11.0,0.0,36.0,0.0,0.0,0.0
62.0,11.0,0.0,41.0,0.0,0.0,0.0
63.0,11.0,1.0,77.0,0.0,0.0,0.0
64.0,11.0,0.0,35.0,0.0,0.0,0.0
65.0,11.0,0.0,64.0,1.0,0.0,1.0
66.0,11.0,0.0,15.0,0.0,0.0,1.0
67.0,12.0,0.0,51.0,0.0,0.0,1.0
68.0,12.0,0.0,50.0,0.0,0.0,0.0
69.0,12.0,1.0,40.0,1.0,0.0,0.0
70.0,12.0,0.0,18.0,1.0,0.0,0.0
71.0,12.0,0.0,45.0,0.0,0.0,1.0
72.0,12.0,0.0,60.0,0.0,0.0,0.0
73.0,13.0,0.0,27.0,1.0,0.0,0.0
74.0,13.0,0.0,56.0,0.0,1.0,1.0
75.0,13.0,0.0,71.0,0.0,0.0,1.0
76.0,13.0,1.0,46.0,1.0,0.0,0.0
77.0,13.0,0.0,56.0,0.0,0.0,0.0
78.0,13.0,0.0,61.0,0.0,0.0,1.0
79.0,14.0,0.0,70.0,1.0,0.0,1.0
80.0,14.0,1.0,73.0,1.0,1.0,1.0
81.0,14.0,0.0,55.0,0.0,1.0,1.0
82.0,14.0,0.0,42.0,1.0,0.0,0.0
83.0,14.0,0.0,32.0,0.0,0.0,0.0
84.0,14.0,0.0,46.0,1.0,0.0,0.0
85.0,15.0,0.0,44.0,0.0,0.0,0.0
86.0,15.0,0.0,64.0,0.0,0.0,0.0
87.0,15.0,0.0,55.0,0.0,0.0,0.0
88.0,15.0,0.0,64.0,1.0,0.0,0.0
89.0,15.0,0.0,87.0,0.0,0.0,1.0
90.0,15.0,1.0,87.0,0.0,1.0,1.0
91.0,16.0,0.0,55.0,1.0,0.0,1.0
92.0,16.0,0.0,48.0,0.0,1.0,0.0
93.0,16.0,0.0,68.0,0.0,0.0,1.0
94.0,16.0,0.0,53.0,0.0,0.0,1.0
95.0,16.0,1.0,87.0,0.0,0.0,0.0
96.0,16.0,0.0,63.0,0.0,1.0,1.0
97.0,17.0,0.0,30.0,0.0,1.0,0.0
98.0,17.0,1.0,72.0,0.0,0.0,1.0
99.0,17.0,0.0,75.0,0.0,0.0,1.0
100.0,17.0,0.0,53.0,0.0,0.0,1.0
101.0,17.0,0.0,42.0,0.0,1.0,0.0
102.0,17.0,0.0,28.0,0.0,0.0,0.0
103.0,18.0,0.0,36.0,0.0,0.0,1.0
104.0,18.0,0.0,33.0,0.0,0.0,1.0
105.0,18.0,0.0,59.0,1.0,1.0,1.0
106.0,18.0,0.0,21.0,0.0,0.0,1.0
107.0,18.0,1.0,42.0,0.0,0.0,0.0
108.0,18.0,0.0,54.0,0.0,0.0,0.0
109.0,19.0,1.0,75.0,0.0,0.0,0.0
110.0,19.0,0.0,49.0,0.0,0.0,1.0
111.0,19.0,0.0,62.0,0.0,1.0,0.0
112.0,19.0,0.0,22.0,1.0,1.0,0.0
113.0,19.0,0.0,53.0,0.0,0.0,0.0
114.0,19.0,0.0,41.0,0.0,1.0,0.0
115.0,20.0,0.0,20.0,0.0,0.0,1.0
116.0,20.0,0.0,72.0,0.0,0.0,0.0
117.0,20.0,0.0,60.0,0.0,1.0,1.0
118.0,20.0,0.0,45.0,1.0,0.0,1.0
119.0,20.0,0.0,43.0,0.0,0.0,0.0
120.0,20.0,1.0,51.0,0.0,0.0,0.0
121.0,21.0,0.0,39.0,0.0,0.0,0.0
122.0,21.0,0.0,57.0,0.0,0.0,0.0
123.0,21.0,1.0,39.0,1.0,1.0,1.0
124.0,21.0,0.0,58.0,0.0,1.0,1.0
125.0,21.0,0.0,58.0,0.0,0.0,1.0
126.0,21.0,0.0,56.0,0.0,0.0,0.0
127.0,22.0,0.0,31.0,0.0,0.0,1.0
128.0,22.0,0.0,68.0,0.0,0.0,1.0
129.0,22.0,0.0,29.0,0.0,1.0,1.0
130.0,22.0,1.0,40.0,0.0,0.0,1.0
131.0,22.0,0.0,23.0,1.0,0.0,0.0
132.0,22.0,0.0,62.0,0.0,1.0,1.0
133.0,23.0,1.0,70.0,1.0,1.0,0.0
134.0,23.0,0.0,63.0,0.0,0.0,1.0
135.0,23.0,0.0,45.0,0.0,0.0,0.0
136.0,23.0,0.0,45.0,0.0,0.0,1.0
137.0,23.0,0.0,43.0,0.0,0.0,1.0
138.0,23.0,0.0,34.0,0.0,1.0,0.0
139.0,24.0,1.0,34.0,0.0,0.0,0.0
140.0,24.0,0.0,32.0,0.0,1.0,0.0
141.0,24.0,0.0,15.0,0.0,0.0,0.0
142.0,24.0,0.0,67.0,0.0,1.0,0.0
143.0,24.0,0.0,31.0,1.0,0.0,0.0
144.0,24.0,0.0,61.0,0.0,1.0,0.0
145.0,25.0,0.0,40.0,0.0,0.0,1.0
146.0,25.0,0.0,28.0,0.0,0.0,0.0
147.0,25.0,1.0,64.0,0.0,0.0,0.0
148.0,25.0,0.0,56.0,0.0,0.0,1.0
149.0,25.0,0.0,54.0,0.0,0.0,1.0
150.0,25.0,0.0,71.0,0.0,0.0,1.0
151.0,26.0,0.0,26.0,0.0,0.0,0.0
152.0,26.0,0.0,29.0,0.0,0.0,1.0
153.0,26.0,1.0,39.0,1.0,0.0,0.0
154.0,26.0,0.0,54.0,0.0,0.0,1.0
155.0,26.0,0.0,50.0,0.0,0.0,1.0
156.0,26.0,0.0,39.0,1.0,0.0,1.0
157.0,27.0,0.0,74.0,1.0,0.0,0.0
158.0,27.0,0.0,39.0,0.0,0.0,0.0
159.0,27.0,0.0,42.0,1.0,1.0,1.0
160.0,27.0,0.0,36.0,0.0,0.0,1.0
161.0,27.0,0.0,66.0,0.0,0.0,1.0
162.0,27.0,1.0,64.0,0.0,0.0,0.0
163.0,28.0,1.0,59.0,0.0,0.0,0.0
164.0,28.0,0.0,43.0,0.0,1.0,1.0
165.0,28.0,0.0,45.0,0.0,0.0,1.0
166.0,28.0,0.0,41.0,0.0,0.0,0.0
167.0,28.0,0.0,37.0,0.0,1.0,1.0
168.0,28.0,0.0,57.0,1.0,0.0,0.0
169.0,29.0,0.0,39.0,0.0,1.0,0.0
170.0,29.0,0.0,31.0,0.0,1.0,1.0
171.0,29.0,0.0,21.0,1.0,1.0,1.0
172.0,29.0,0.0,41.0,1.0,0.0,1.0
173.0,29.0,0.0,45.0,0.0,0.0,0.0
174.0,29.0,1.0,35.0,0.0,1.0,0.0
""")

## Helper: collinear_columns

A step that no library does, written out. In the code download it is the file `collinear_columns.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Which explanatory variables are exact linear combinations of the ones before them.

Such a variable adds no information and makes the model impossible to estimate. Each
column is centred and scaled to length 1, then regressed on the columns already kept;
if nothing is left over (squared residual length at most 1e-10) the column is
redundant. A constant column is redundant too. The *later* member of a redundant pair
is the one reported, so a variable listed first survives.
"""

import numpy as np


def collinear_columns(values, names, tolerance=1e-10):
    centred = values - values.mean(axis=0)
    scale = np.linalg.norm(centred, axis=0)
    scale[scale == 0] = 1.0
    centred = centred / scale
    dropped, kept = [], []
    for j, name in enumerate(names):
        column = centred[:, j]
        if kept:
            basis = centred[:, kept]
            column = column - basis @ np.linalg.lstsq(basis, column, rcond=None)[0]
        if column @ column <= tolerance or values[:, j].max() == values[:, j].min():
            dropped.append(name)
        else:
            kept.append(j)
    return dropped

## Helper: conditional_logistic

A step that no library does, written out. In the code download it is the file `conditional_logistic.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Two steps of conditional logistic regression statsmodels does not print.

Within a matched set holding m cases, every arrangement of which members are the
cases has a probability proportional to the product of their exp(x'b). Two numbers
come out of that:

* with no explanatory variable every arrangement is equally likely, so the set's
  likelihood is 1 / C(size, m) -- the baseline the model is tested against;
* a member's probability of being one of the m cases is the sum over the
  arrangements that include it, divided by the sum over all of them. Those sums are
  elementary symmetric polynomials of the weights exp(x'b).
"""

from math import comb, log

import numpy as np


def null_log_likelihood(cases, groups):
    """Conditional log-likelihood with no explanatory variable."""
    total = 0.0
    for group in np.unique(groups):
        size, positive = int((groups == group).sum()), int(cases[groups == group].sum())
        if 0 < positive < size:
            total -= log(comb(size, positive))
    return total


def elementary_symmetric(weights, order):
    """e_0 ... e_order of the weights: e_k sums the products of every k of them."""
    out = np.zeros(order + 1)
    out[0] = 1.0
    for weight in weights:
        for k in range(order, 0, -1):
            out[k] += out[k - 1] * weight
    return out


def case_probabilities(linear, cases, groups):
    """Each member's probability of being one of its set's cases, given x'b."""
    probability = np.zeros(len(linear))
    for group in np.unique(groups):
        index = np.flatnonzero(groups == group)
        m = int(cases[index].sum())
        if m <= 0 or m >= len(index):
            probability[index] = m / len(index)
            continue
        weights = np.exp(linear[index] - linear[index].max())  # scaled, to avoid overflow
        denominator = elementary_symmetric(weights, m)[m]
        for position, member in enumerate(index):
            rest = np.delete(weights, position)
            probability[member] = weights[position] * elementary_symmetric(rest, m - 1)[m - 1] / denominator
    return probability

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.discrete.conditional_models import ConditionalLogit
from statsmodels.tools.numdiff import approx_fprime
# collinear_columns: defined in the helper cell collinear_columns above (collinear_columns.py in the code download)
# case_probabilities and null_log_likelihood: defined in the helper cell conditional_logistic above (conditional_logistic.py in the code download)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# The response is coded 0/1; the stratum column says which matched set each row is in.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Churned"
predictors = ["Monthly bill ($)", "Filed complaint", "Contacted support", "Annual contract"]
stratum = "Matched set"
frame = data[[response, *predictors]].apply(pd.to_numeric, errors="coerce")
max_iter = 50
confidence = 95 / 100


# A stratum as text; a number read as 2.0 prints as "2".
def label(value):
    return str(int(value)) if isinstance(value, float) and value.is_integer() else str(value)

## Cases

In [ ]:
r, x, s = frame[response].notna(), frame[predictors].notna().all(axis=1), data[stratum].notna()
summary = pd.DataFrame({"n": [
    (r & x & s).sum(), (~r & x & s).sum(), (r & ~x & s).sum(), (r & x & ~s).sum(),
    (r & ~x & ~s).sum(), (~r & x & ~s).sum(), (~r & ~x & s).sum(), (~r & ~x & ~s).sum(), len(frame),
]}, index=["valid", "response missing", "explanatory missing", "stratum missing",
           "only the response", "only the explanatory", "only the stratum", "all missing", "total"])
summary["%"] = summary["n"] / len(frame)
print(summary)
valid = frame[r & x & s]
y = valid[response].to_numpy(float)
strata = data.loc[r & x & s, stratum].map(label).to_numpy()
# Strata are numbered in the order they first appear.
stratum_levels = list(dict.fromkeys(strata))
groups = np.array([stratum_levels.index(v) for v in strata])
n = len(y)

## Descriptive statistics and correlations, overall and by outcome

In [ ]:
parts = {"all": valid, f"{response}=0": valid[y == 0], f"{response}=1": valid[y == 1]}
descriptive = pd.concat({
    name: pd.DataFrame({"n": len(part), "mean": part[predictors].mean(),
                        "variance": part[predictors].var(ddof=1), "SD": part[predictors].std(ddof=1),
                        "min": part[predictors].min(), "max": part[predictors].max()})
    for name, part in parts.items()
})
correlation = pd.concat({name: part[predictors].corr() for name, part in parts.items()})
print()
print(descriptive)
print()
print(correlation)

## Outcome by stratum

In [ ]:
by_stratum = pd.crosstab(strata, y.astype(int)).reindex(index=stratum_levels, columns=[0, 1],
                                                         fill_value=0)
by_stratum.loc["total"] = by_stratum.sum()
by_stratum["share 0"] = by_stratum[0] / by_stratum[[0, 1]].sum(axis=1)
by_stratum["share 1"] = by_stratum[1] / by_stratum[[0, 1]].sum(axis=1)
by_stratum["n"] = by_stratum[[0, 1]].sum(axis=1)
by_stratum["%"] = by_stratum["n"] / n
print()
print(by_stratum)

## Variables that are linear combinations of others

In [ ]:
values = valid[predictors].to_numpy(float)
dropped = collinear_columns(values, predictors)
print("\nLinearly dependent:", dropped or "none")
# They are left out of the model.
predictors = [name for name in predictors if name not in dropped]
values = valid[predictors].to_numpy(float)

## The model

In [ ]:
# Exact conditional likelihood: each stratum contributes only which of its members are
# the cases, so anything constant within a stratum drops out. Newton-Raphson from
# zero, as in the app (the quasi-Newton default can stop at the starting point).
# statsmodels takes the Hessian by forward differences of its exact score, good to about
# 1e-8; central differences, as the app takes them, are good to about 1e-10, and the
# standard errors come from that Hessian.


class CentredConditionalLogit(ConditionalLogit):
    def hessian(self, params):
        return np.atleast_2d(approx_fprime(params, self.score, centered=True))


def fit(columns):
    return CentredConditionalLogit(y, columns, groups=groups).fit(
        method="newton", disp=0, maxiter=max_iter * 10, start_params=np.zeros(columns.shape[1]))


model = fit(values)
deviance = -2 * model.llf
null_deviance = -2 * null_log_likelihood(y, groups)
chi2 = null_deviance - deviance
df = len(predictors)
model_tests = pd.DataFrame([[null_deviance, np.nan, np.nan, np.nan],
                            [deviance, chi2, df, stats.chi2.sf(chi2, df)]],
                           index=["without the variables", "with the variables"],
                           columns=["-2LL", "chi-square", "df", "P"])
print()
print(model_tests)

# Each variable's likelihood-ratio test: refit without it, on 1 df.
rows = []
for j in range(len(predictors)):
    others = np.delete(values, j, axis=1)
    reduced = -2 * fit(others).llf if others.shape[1] else null_deviance
    rows.append([reduced, deviance, reduced - deviance, 1, stats.chi2.sf(reduced - deviance, 1)])
variable_tests = pd.DataFrame(rows, index=predictors,
                              columns=["reduced -2LL", "final -2LL", "chi-square", "df", "P"])
print()
print(variable_tests)

## Coefficients

In [ ]:
z = stats.norm.ppf(0.5 + confidence / 2)
b, se = np.asarray(model.params), np.asarray(model.bse)
coefficients = pd.DataFrame({
    "b": b, "SE": se, "lower": b - z * se, "upper": b + z * se,
    "odds ratio": np.exp(b), "OR lower": np.exp(b - z * se), "OR upper": np.exp(b + z * se),
    "Wald": (b / se) ** 2, "df": 1, "P": stats.chi2.sf((b / se) ** 2, 1),
}, index=predictors)
covariance = np.asarray(model.cov_params())
covariance = pd.DataFrame((covariance + covariance.T) / 2, index=predictors, columns=predictors)
print()
print(coefficients)
print()
print(covariance)

## Classification table

In [ ]:
# Each member's probability of being one of its stratum's cases, from the exact
# conditional model; a member is predicted a case when that is at least 0.5.
probability = case_probabilities(values @ b, y, groups)
predicted = (probability >= 0.5).astype(int)
classification = pd.crosstab(y.astype(int), predicted).reindex(index=[0, 1], columns=[0, 1],
                                                               fill_value=0)
classification["hit rate"] = np.diag(classification) / classification.sum(axis=1)
hit_rate = (predicted == y).mean()
print()
print(classification)
print(f"overall hit rate: {hit_rate:.4f}")

## Predicted probabilities

In [ ]:
predictions = pd.DataFrame({"no": np.arange(1, n + 1), "observed": y, "predicted": probability,
                            "residual": y - probability})
print()
print(predictions.to_string(index=False))

## What the result says

Conditional logistic regression compares cases inside matched sets. Here each of 29 sets holds one customer who cancelled and five who stayed, all from the same signup week.

The odds ratio says how the odds of being the one who leaves multiply when a predictor rises by one. A filed complaint has an odds ratio of 4.75 (P = 0.003; a small P value means chance alone would rarely give such a link). An annual contract has 0.26, so it goes with much lower odds of leaving (P = 0.009). Each extra dollar on the monthly bill has 1.05 (P = 0.001). Contacting support has 1.56, but P = 0.375, and its confidence interval, the range of values the data fit, runs from 0.58 to 4.18. No link is shown there.

So the customer who left usually had a higher bill, had filed a complaint, and was not on an annual contract.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.